
# RPM del buzo a partir del video Slo‑mo del iPhone 15

Este notebook usa **dos métodos independientes**:

1. **Método principal: orientación del eje del buzo.**  
   Se segmenta el buzo blanco, se obtiene el eje largo con `minAreaRect` y se sigue su ángulo cuadro a cuadro.

2. **Control independiente: periodicidad de intensidad + FFT.**  
   Se observa una pequeña zona lateral. Como una barra casi simétrica pasa dos veces por esa zona por cada vuelta, aparece un pico aproximadamente en \(2 f_{\rm rot}\).

## Punto clave sobre el tiempo

Para este video Slo‑mo no hay que usar los \(30\ \mathrm{fps}\) del archivo como tiempo físico durante la parte lenta.

Si la captura fue a \(240\ \mathrm{fps}\), dentro de la zona lenta:

\[
\Delta t_{\rm físico}=\frac{1}{240}\ {\rm s}
\]

El archivo puede reproducirse a \(30\ \mathrm{fps}\), por lo que esa parte queda estirada aproximadamente un factor

\[
\frac{240}{30}=8.
\]

Por eso el comienzo y el final pueden verse a velocidad normal, mientras que la zona central ocupa mucho más tiempo en el archivo.


In [ ]:

# =========================
# PARÁMETROS QUE PODÉS TOCAR
# =========================

from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH = Path(
    r"C:\Users\user\Desktop\Buzo l5\buzo_dia_3\Pastilla_2.mov"
)

# FPS físicos de captura en la parte Slo-mo
FPS_FISICOS_SLOW = 240.0

# Para ESTE archivo conviene usar una zona bien metida dentro de la parte lenta.
SLOW_START_PLAYBACK_S = 5.0
SLOW_END_PLAYBACK_S   = 27.0

# ROI automática, expresada como fracción de la imagen:
# (x1, y1, x2, y2)
# Está elegida para el video que me pasaste.
ROI_FRAC = (0.14, 0.47, 0.86, 0.78)

# Si querés seleccionar la ROI con el mouse, ponelo en True.
USE_INTERACTIVE_ROI = False

# Umbral de gris.
# None = se calcula automáticamente con un percentil del frame de referencia.
THRESH_GRAY = None
THRESH_PERCENTILE = 90

# Morfología para unir el cuerpo blanco del buzo
MORPH_KERNEL = 9
MORPH_ITER = 2

# Para estimar repetibilidad: 60 frames físicos = 0.25 s reales a 240 fps
BLOCK_FRAMES = 60

# En la FFT, una barra aproximadamente simétrica genera 2 máximos por vuelta
PASSES_PER_REV = 2


In [ ]:

# =========================
# 1) METADATOS DEL VIDEO
# =========================

if not VIDEO_PATH.exists():
    raise FileNotFoundError(
        f"No encuentro el archivo:\n{VIDEO_PATH}\n\n"
        "Revisá solamente VIDEO_PATH en la celda de parámetros."
    )

cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise RuntimeError("OpenCV no pudo abrir el video.")

fps_archivo = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duracion_playback = n_frames / fps_archivo

print(f"FPS informados por el archivo: {fps_archivo:.3f}")
print(f"Cantidad de frames: {n_frames}")
print(f"Resolución leída por OpenCV: {width} x {height}")
print(f"Duración de reproducción: {duracion_playback:.3f} s")

factor_slow = FPS_FISICOS_SLOW / fps_archivo
print(f"\nFactor captura/reproducción en la parte lenta: {factor_slow:.3f}x")

print(
    "\nIMPORTANTE: durante la parte lenta se usará "
    f"dt = 1/{FPS_FISICOS_SLOW:.0f} s, NO 1/{fps_archivo:.0f} s."
)

cap.release()


In [ ]:

# =========================
# 2) ELEGIR / VER LA ROI
# =========================

cap = cv2.VideoCapture(str(VIDEO_PATH))

t_ref = 0.5 * (SLOW_START_PLAYBACK_S + SLOW_END_PLAYBACK_S)
frame_ref_idx = int(round(t_ref * fps_archivo))

cap.set(cv2.CAP_PROP_POS_FRAMES, frame_ref_idx)
ok, frame_ref = cap.read()
cap.release()

if not ok:
    raise RuntimeError("No pude leer el frame de referencia.")

H, W = frame_ref.shape[:2]

if USE_INTERACTIVE_ROI:
    roi_cv = cv2.selectROI(
        "Selecciona ROI grande alrededor del buzo y presiona ENTER",
        frame_ref,
        showCrosshair=True,
        fromCenter=False,
    )
    cv2.destroyAllWindows()

    x, y, w, h = map(int, roi_cv)

else:
    fx1, fy1, fx2, fy2 = ROI_FRAC

    x = int(round(fx1 * W))
    y = int(round(fy1 * H))
    w = int(round((fx2 - fx1) * W))
    h = int(round((fy2 - fy1) * H))

ROI = (x, y, w, h)

print("ROI =", ROI)

frame_rgb = cv2.cvtColor(frame_ref, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(7, 10))
plt.imshow(frame_rgb)
plt.gca().add_patch(
    plt.Rectangle(
        (x, y),
        w,
        h,
        fill=False,
        linewidth=2,
    )
)
plt.title("Frame de referencia + ROI")
plt.axis("off")
plt.show()


In [ ]:

# =========================
# 3) FUNCIONES DE DETECCIÓN
# =========================

def wrap_line_delta(delta):
    """
    Diferencia angular para una orientación de eje, que es periódica en pi.
    Devuelve el incremento en el intervalo [-pi/2, pi/2].
    """
    return 0.5 * np.arctan2(
        np.sin(2.0 * delta),
        np.cos(2.0 * delta),
    )


def preparar_mascara_central(w, h):
    yy, xx = np.mgrid[0:h, 0:w]

    cx = w / 2.0
    cy = h / 2.0

    radio = 0.50 * min(w, h)

    mascara = (
        (xx - cx) ** 2
        + (yy - cy) ** 2
        <= radio**2
    )

    return mascara


def detectar_orientacion(
    frame,
    ROI,
    threshold_gray,
    devolver_debug=False,
):
    """
    Segmenta el buzo blanco dentro de la ROI y obtiene el eje largo.

    Retorna:
        angle : orientación del eje en [0, pi)
        aspect: relación largo/ancho del rectángulo ajustado
        debug : opcional
    """

    x, y, w, h = ROI

    crop = frame[
        y:y + h,
        x:x + w,
    ]

    gray = cv2.cvtColor(
        crop,
        cv2.COLOR_BGR2GRAY,
    )

    central = preparar_mascara_central(
        w,
        h,
    )

    mask = (
        (gray > threshold_gray)
        & central
    ).astype(np.uint8) * 255

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (MORPH_KERNEL, MORPH_KERNEL),
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=MORPH_ITER,
    )

    n_labels, labels, stats, centroids = (
        cv2.connectedComponentsWithStats(mask)
    )

    cx0 = w / 2.0
    cy0 = h / 2.0

    candidates = []

    min_area = 0.004 * w * h
    max_center_dist = 0.30 * min(w, h)

    for k in range(1, n_labels):

        _, _, _, _, area = stats[k]

        cx, cy = centroids[k]

        dist = np.hypot(
            cx - cx0,
            cy - cy0,
        )

        if (
            area > min_area
            and dist < max_center_dist
        ):
            candidates.append(
                (area, k)
            )

    if not candidates:
        if devolver_debug:
            return np.nan, np.nan, {
                "crop": crop,
                "mask": mask,
                "box": None,
            }

        return np.nan, np.nan

    _, k_best = max(candidates)

    ys, xs = np.where(
        labels == k_best
    )

    pts = np.column_stack(
        [xs, ys]
    ).astype(np.float32)

    rect = cv2.minAreaRect(pts)
    box = cv2.boxPoints(rect)

    edge_vectors = [
        box[(j + 1) % 4] - box[j]
        for j in range(4)
    ]

    edge_lengths = [
        np.linalg.norm(v)
        for v in edge_vectors
    ]

    v_long = edge_vectors[
        int(np.argmax(edge_lengths))
    ]

    angle = np.arctan2(
        v_long[1],
        v_long[0],
    ) % np.pi

    rw, rh = rect[1]

    aspect = max(rw, rh) / max(
        min(rw, rh),
        1e-9,
    )

    if devolver_debug:
        return angle, aspect, {
            "crop": crop,
            "mask": mask,
            "box": box,
        }

    return angle, aspect


In [ ]:

# =========================
# 4) ELEGIR UMBRAL Y VALIDAR VISUALMENTE
# =========================

x, y, w, h = ROI

gray_ref = cv2.cvtColor(
    frame_ref[
        y:y + h,
        x:x + w,
    ],
    cv2.COLOR_BGR2GRAY,
)

if THRESH_GRAY is None:
    threshold_gray = float(
        np.percentile(
            gray_ref,
            THRESH_PERCENTILE,
        )
    )
else:
    threshold_gray = float(
        THRESH_GRAY
    )

print(
    f"Umbral usado = {threshold_gray:.1f}"
)

angle_ref, aspect_ref, dbg = detectar_orientacion(
    frame_ref,
    ROI,
    threshold_gray,
    devolver_debug=True,
)

crop_rgb = cv2.cvtColor(
    dbg["crop"],
    cv2.COLOR_BGR2RGB,
)

overlay = crop_rgb.copy()

if dbg["box"] is not None:
    box_int = np.int32(
        dbg["box"]
    )

    cv2.polylines(
        overlay,
        [box_int],
        isClosed=True,
        color=(255, 0, 0),
        thickness=3,
    )

fig = plt.figure(figsize=(12, 5))

ax1 = fig.add_subplot(1, 2, 1)
ax1.imshow(overlay)
ax1.set_title(
    f"Buzo detectado | aspect={aspect_ref:.2f}"
)
ax1.axis("off")

ax2 = fig.add_subplot(1, 2, 2)
ax2.imshow(
    dbg["mask"],
    cmap="gray",
)
ax2.set_title("Máscara usada")
ax2.axis("off")

plt.show()

print(
    "Chequeo visual: el rectángulo debe seguir al buzo, "
    "no a un reflejo del vaso."
)


In [ ]:

# =========================
# 5) TRACKEAR LA ORIENTACIÓN EN TODO EL VIDEO
# =========================

cap = cv2.VideoCapture(
    str(VIDEO_PATH)
)

angles = []
aspects = []

for i in range(n_frames):

    ok, frame = cap.read()

    if not ok:
        break

    angle, aspect = detectar_orientacion(
        frame,
        ROI,
        threshold_gray,
        devolver_debug=False,
    )

    angles.append(angle)
    aspects.append(aspect)

cap.release()

angles = np.asarray(
    angles,
    dtype=float,
)

aspects = np.asarray(
    aspects,
    dtype=float,
)

if len(angles) < 2:
    raise RuntimeError(
        "No se pudo obtener una trayectoria angular."
    )

dtheta = wrap_line_delta(
    np.diff(angles)
)

t_playback_dtheta = (
    np.arange(len(dtheta))
    / fps_archivo
)

window = 21

kernel = np.ones(
    window
) / window

dtheta_filled = np.where(
    np.isfinite(dtheta),
    dtheta,
    0.0,
)

valid = np.isfinite(
    dtheta
).astype(float)

num = np.convolve(
    dtheta_filled,
    kernel,
    mode="same",
)

den = np.convolve(
    valid,
    kernel,
    mode="same",
)

rolling = np.divide(
    num,
    den,
    out=np.full_like(num, np.nan),
    where=den > 0,
)

plt.figure(figsize=(12, 5))

plt.plot(
    t_playback_dtheta,
    np.degrees(dtheta),
    alpha=0.30,
    label="incremento por frame",
)

plt.plot(
    t_playback_dtheta,
    np.degrees(rolling),
    linewidth=2,
    label="promedio móvil",
)

plt.axvline(
    SLOW_START_PLAYBACK_S,
    linestyle="--",
)

plt.axvline(
    SLOW_END_PLAYBACK_S,
    linestyle="--",
)

plt.xlabel(
    "Tiempo de reproducción del archivo [s]"
)

plt.ylabel(
    "Cambio de orientación del eje [deg/frame]"
)

plt.title(
    "Orientación del buzo: la zona entre líneas se usa para calcular RPM"
)

plt.grid()
plt.legend()
plt.show()

print(
    "Fracción de frames con orientación detectada:",
    f"{np.isfinite(angles).mean():.3f}",
)

print(
    "Aspect ratio mediano del objeto detectado:",
    f"{np.nanmedian(aspects):.2f}",
)


In [ ]:

# =========================
# 6) RPM POR ORIENTACIÓN
# =========================

i0 = int(
    round(
        SLOW_START_PLAYBACK_S
        * fps_archivo
    )
)

i1 = int(
    round(
        SLOW_END_PLAYBACK_S
        * fps_archivo
    )
)

i0 = max(
    0,
    min(i0, len(angles) - 2),
)

i1 = max(
    i0 + 2,
    min(i1, len(angles) - 1),
)

dtheta_slow = dtheta[
    i0:i1
]

mean_dtheta = np.nanmean(
    dtheta_slow
)

omega_rad_s = (
    mean_dtheta
    * FPS_FISICOS_SLOW
)

rpm_orientacion = (
    abs(omega_rad_s)
    * 60.0
    / (2.0 * np.pi)
)

rpm_blocks = []

for j in range(
    0,
    len(dtheta_slow) - BLOCK_FRAMES + 1,
    BLOCK_FRAMES,
):

    block = dtheta_slow[
        j:j + BLOCK_FRAMES
    ]

    rpm_b = (
        abs(np.nanmean(block))
        * FPS_FISICOS_SLOW
        * 60.0
        / (2.0 * np.pi)
    )

    rpm_blocks.append(rpm_b)

rpm_blocks = np.asarray(
    rpm_blocks,
    dtype=float,
)

rpm_blocks_std = (
    np.nanstd(
        rpm_blocks,
        ddof=1,
    )
    if len(rpm_blocks) > 1
    else np.nan
)

cum_angle = np.r_[
    0.0,
    np.nancumsum(dtheta_slow),
]

t_phys = (
    np.arange(len(cum_angle))
    / FPS_FISICOS_SLOW
)

turns = (
    cum_angle
    / (2.0 * np.pi)
)

plt.figure(figsize=(10, 5))
plt.plot(
    t_phys,
    turns,
)
plt.xlabel(
    "Tiempo físico dentro de la zona lenta [s]"
)
plt.ylabel(
    "Vueltas acumuladas"
)
plt.title(
    "Validación: las vueltas acumuladas deberían crecer casi linealmente"
)
plt.grid()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(
    rpm_blocks,
    marker="o",
)
plt.axhline(
    rpm_orientacion,
    linestyle="--",
)
plt.xlabel(
    "Bloque"
)
plt.ylabel(
    "RPM"
)
plt.title(
    "RPM estimadas por bloques"
)
plt.grid()
plt.show()

print(
    f"RPM por orientación = {rpm_orientacion:.1f} rpm"
)

if len(rpm_blocks) > 1:
    print(
        "Dispersión entre bloques "
        f"(repetibilidad, no error absoluto) = "
        f"{rpm_blocks_std:.1f} rpm"
    )

print(
    f"Tiempo físico realmente analizado = "
    f"{len(dtheta_slow) / FPS_FISICOS_SLOW:.3f} s"
)



## Por qué el promedio de los incrementos funciona

El eje de una barra es equivalente después de \(180^\circ\). Por eso se trabaja con el ángulo módulo \(\pi\), no módulo \(2\pi\).

Para dos frames consecutivos se calcula

\[
\Delta\theta_i=
\frac{1}{2}
\operatorname{atan2}
\left[
\sin(2\Delta\theta),
\cos(2\Delta\theta)
\right].
\]

En la parte lenta, a \(240\ \mathrm{fps}\), el giro entre frames es suficientemente pequeño como para evitar aliasing. Luego,

\[
\omega \simeq
\frac{\langle \Delta\theta\rangle}{1/240}
\]

y

\[
{\rm RPM}
=
\frac{|\omega|}{2\pi}\,60.
\]


In [ ]:

# =========================
# 7) CONTROL INDEPENDIENTE: FFT DE INTENSIDAD
# =========================

x, y, w, h = ROI

px1 = int(
    0.64 * w
)

px2 = int(
    0.78 * w
)

py1 = int(
    0.44 * h
)

py2 = int(
    0.56 * h
)

crop_ref = frame_ref[
    y:y + h,
    x:x + w,
].copy()

crop_ref_rgb = cv2.cvtColor(
    crop_ref,
    cv2.COLOR_BGR2RGB,
)

plt.figure(figsize=(8, 6))
plt.imshow(crop_ref_rgb)

plt.gca().add_patch(
    plt.Rectangle(
        (px1, py1),
        px2 - px1,
        py2 - py1,
        fill=False,
        linewidth=2,
    )
)

plt.title(
    "Parche usado para la señal periódica"
)
plt.axis("off")
plt.show()

cap = cv2.VideoCapture(
    str(VIDEO_PATH)
)

signal = []

for i in range(n_frames):

    ok, frame = cap.read()

    if not ok:
        break

    if i0 <= i < i1:

        crop = frame[
            y:y + h,
            x:x + w,
        ]

        gray = cv2.cvtColor(
            crop,
            cv2.COLOR_BGR2GRAY,
        )

        patch = gray[
            py1:py2,
            px1:px2,
        ]

        signal.append(
            patch.mean()
        )

cap.release()

signal = np.asarray(
    signal,
    dtype=float,
)

signal = signal - np.mean(
    signal
)

window_fft = np.hanning(
    len(signal)
)

spectrum = np.abs(
    np.fft.rfft(
        signal * window_fft
    )
)

freq = np.fft.rfftfreq(
    len(signal),
    d=1.0 / FPS_FISICOS_SLOW,
)

fmin = 5.0
fmax = min(
    100.0,
    0.49 * FPS_FISICOS_SLOW,
)

valid_f = (
    (freq >= fmin)
    & (freq <= fmax)
)

candidate_idx = np.where(
    valid_f
)[0]

k_peak = candidate_idx[
    np.argmax(
        spectrum[
            candidate_idx
        ]
    )
]

if (
    1 <= k_peak < len(spectrum) - 1
):

    a0 = spectrum[
        k_peak - 1
    ]

    a1 = spectrum[
        k_peak
    ]

    a2 = spectrum[
        k_peak + 1
    ]

    denom = (
        a0
        - 2.0 * a1
        + a2
    )

    if abs(denom) > 1e-12:
        delta_bin = (
            0.5
            * (a0 - a2)
            / denom
        )
    else:
        delta_bin = 0.0

else:
    delta_bin = 0.0

df = freq[1] - freq[0]

f_peak = (
    freq[k_peak]
    + delta_bin * df
)

rpm_fft = (
    f_peak
    * 60.0
    / PASSES_PER_REV
)

plt.figure(figsize=(10, 5))

plt.plot(
    freq,
    spectrum,
)

plt.axvline(
    f_peak,
    linestyle="--",
    label=f"pico = {f_peak:.2f} Hz",
)

plt.xlim(
    0,
    min(
        80,
        FPS_FISICOS_SLOW / 2,
    ),
)

plt.xlabel(
    "Frecuencia [Hz]"
)

plt.ylabel(
    "Amplitud FFT [u.a.]"
)

plt.title(
    "Espectro de la señal de intensidad"
)

plt.grid()
plt.legend()
plt.show()

print(
    f"Pico de intensidad = {f_peak:.3f} Hz"
)

print(
    f"RPM por FFT = {rpm_fft:.1f} rpm"
)

print(
    f"(Se dividió por {PASSES_PER_REV} porque "
    "la barra casi simétrica produce "
    "dos pasadas equivalentes por vuelta.)"
)


In [ ]:

# =========================
# 8) COMPARACIÓN FINAL
# =========================

difference = abs(
    rpm_orientacion
    - rpm_fft
)

relative_difference = (
    difference
    / (
        0.5
        * (
            rpm_orientacion
            + rpm_fft
        )
    )
    * 100.0
)

rpm_final = (
    0.5
    * (
        rpm_orientacion
        + rpm_fft
    )
)

print(
    f"Método orientación : {rpm_orientacion:.1f} rpm"
)

print(
    f"Método FFT         : {rpm_fft:.1f} rpm"
)

print(
    f"Diferencia         : {difference:.1f} rpm "
    f"({relative_difference:.2f} %)"
)

print(
    f"\nValor combinado orientativo: {rpm_final:.1f} rpm"
)

if relative_difference < 3.0:
    print(
        "\nLos dos métodos son consistentes entre sí."
    )
else:
    print(
        "\nLos métodos difieren más de lo deseable. "
        "Revisá ROI, máscara, límites de la zona lenta "
        "y el parche de la FFT."
    )



## Nota para el video normal de \(60\ \mathrm{fps}\)

Si el buzo está cerca de \(1000\ \mathrm{rpm}\), una barra simétrica queda **cerca o por encima del límite de aliasing** para un seguimiento de eje a \(60\ \mathrm{fps}\).

En ese segundo video conviene aprovechar una **marca asimétrica concreta de uno de los extremos** para seguir una orientación de \(360^\circ\), no solamente el eje de \(180^\circ\).

Por eso, para medir la frecuencia alta, el video de \(240\ \mathrm{fps}\) es en principio el más seguro.
